In [3]:
import os
import time
import pandas as pd

# --- [중요] OS 에러(SSL 인증서 경로) 방지 설정 ---
# 시스템에 잘못 설정된 인증서 경로를 무시하도록 설정합니다.
os.environ['WDM_SSL_VERIFY'] = '0' 
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
# ----------------------------------------------

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager

def crawl_paikdabang():
    # 브라우저 설정
    chrome_options = Options()
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    
    # 7. 가상 접속 보이게 설정 (Headless 모드 사용 안 함)
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    url = "https://paikdabang.com/store/"
    driver.get(url)
    
    store_data = []
    current_page = 1

    try:
        while True:
            # 5. 페이지 넘어갈 때마다 콘솔 출력
            print(f"--- 현재 {current_page} 페이지 수집 중... ---")
            
            # 1. store_list가 나타날 때까지 대기
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "store_list"))
            )
            time.sleep(1.5) # 안정적인 로딩 대기

            # BeautifulSoup 파싱
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            
            # 1. id="store_list" 안의 class="tr_list" 수집
            rows = soup.select("#store_list .tr_list")
            
            if not rows:
                print("더 이상 수집할 데이터가 없습니다.")
                break

            for row in rows:
                # 2. 클래스명에 맞춰 컬럼명 매핑
                # 지역, 매장명, 주소, 전화번호
                area = row.find(class_='td_area').get_text(strip=True) if row.find(class_='td_area') else ""
                shop = row.find(class_='td_shop').get_text(strip=True) if row.find(class_='td_shop') else ""
                addr = row.find(class_='td_addr').get_text(strip=True) if row.find(class_='td_addr') else ""
                tel = row.find(class_='td_tel').get_text(strip=True) if row.find(class_='td_tel') else ""
                
                store_data.append({
                    "지역": area,
                    "매장명": shop,
                    "주소": addr,
                    "전화번호": tel
                })

            # 3~4. 페이지네이션 처리 (다음 페이지 클릭)
            try:
                next_page_num = current_page + 1
                # li 태그 중 data-page 속성이 다음 번호인 요소를 찾음
                next_button = driver.find_elements(By.CSS_SELECTOR, f"#pagination li[data-page='{next_page_num}']")
                
                if next_button:
                    # 셀레늄 클릭이 안 먹힐 경우를 대비해 스크립트로 클릭
                    driver.execute_script("arguments[0].click();", next_button[0])
                    current_page += 1
                else:
                    print("마지막 페이지까지 수집을 완료했습니다.")
                    break
            except Exception as e:
                print(f"페이지 이동 중 알 수 없는 종료: {e}")
                break

    finally:
        # 6. CSV 저장 및 OS 에러(권한) 방지
        if store_data:
            df = pd.DataFrame(store_data)
            try:
                df.to_csv("paikdabang.csv", index=False, encoding="utf-8-sig")
                print(f"\n[성공] 총 {len(df)}개 지점 저장 완료: paikdabang.csv")
            except OSError:
                # 파일이 열려있어서 저장 안 될 경우 대비
                alt_name = f"paikdabang_{int(time.time())}.csv"
                df.to_csv(alt_name, index=False, encoding="utf-8-sig")
                print(f"\n[알림] 기존 파일이 열려있어 다른 이름으로 저장했습니다: {alt_name}")
        
        driver.quit()

if __name__ == "__main__":
    crawl_paikdabang()

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'googlechromelabs.github.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'googlechromelabs.github.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


--- 현재 1 페이지 수집 중... ---
--- 현재 2 페이지 수집 중... ---
--- 현재 3 페이지 수집 중... ---
--- 현재 4 페이지 수집 중... ---
--- 현재 5 페이지 수집 중... ---
--- 현재 6 페이지 수집 중... ---
--- 현재 7 페이지 수집 중... ---
--- 현재 8 페이지 수집 중... ---
--- 현재 9 페이지 수집 중... ---
--- 현재 10 페이지 수집 중... ---
--- 현재 11 페이지 수집 중... ---
--- 현재 12 페이지 수집 중... ---
--- 현재 13 페이지 수집 중... ---
--- 현재 14 페이지 수집 중... ---
--- 현재 15 페이지 수집 중... ---
--- 현재 16 페이지 수집 중... ---
--- 현재 17 페이지 수집 중... ---
--- 현재 18 페이지 수집 중... ---
--- 현재 19 페이지 수집 중... ---
--- 현재 20 페이지 수집 중... ---
--- 현재 21 페이지 수집 중... ---
--- 현재 22 페이지 수집 중... ---
--- 현재 23 페이지 수집 중... ---
--- 현재 24 페이지 수집 중... ---
--- 현재 25 페이지 수집 중... ---
--- 현재 26 페이지 수집 중... ---
--- 현재 27 페이지 수집 중... ---
--- 현재 28 페이지 수집 중... ---
--- 현재 29 페이지 수집 중... ---
--- 현재 30 페이지 수집 중... ---
--- 현재 31 페이지 수집 중... ---
--- 현재 32 페이지 수집 중... ---
--- 현재 33 페이지 수집 중... ---
--- 현재 34 페이지 수집 중... ---
--- 현재 35 페이지 수집 중... ---
--- 현재 36 페이지 수집 중... ---
--- 현재 37 페이지 수집 중... ---
--- 현재 38 페이지 수집 중... ---
--- 현재 39 페이지 수집 중...